In [248]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [335]:
df = pd.read_csv("../raw-data/20260309-ges-obs.csv") # Observations of round 0 GES's 
prev_dataset = pd.read_csv('../raw-data/20260305-round-0-allcols.csv', skiprows=[1]) # Data used to generate round 0 GES's

In [336]:
ges_known = pd.read_csv('../raw-data/20260305-ges-known.csv', skiprows=[1]) ## Since we could not utilize "known GES"
ges_known.rename({'ID':'experiment_id'}, axis=1, inplace=True)
ges_known.drop('Note',axis=1, inplace=True)
ges_known.experiment_id = 'round0_' + ges_known.experiment_id
ges_known['[PEG4K 40%] (%)'] = np.maximum(ges_known['[PEG4K 40%] (%)'], 0)
ges_known.IsNEB = ges_known.IsNEB.astype(int)
ges_known.success = ges_known.success.astype(int)
ges_known.isplamGFP = ges_known.isplamGFP.astype(int)

In [337]:
ges_known

,experiment_id,[DNA] (nM),[PEG4K 40%] (%),[RNAse Inhib] (U/mL),IsNEB,[PMix] (mg/mL),[Ribosome] (uM),Rxn Volume (uL),[Magnesium acetate] (mM),[Creatine phosphate] (mM),[PPK] (uM),[PolyP] (mM),[Potassium glutamate] (mM),sigmoid_steady_state (ng/uL),sigmoid_rate (1/h),sigmoid_time_offset (h),success,isplamGFP
0,round0_GES-7,1.000000,0.0,2000.000003,0,1.767172,1.8,10.0,0.150808,100.000001,0.0,0.0,60.000001,0,NaN,NaN,0,0
1,round0_GES-9,19.999999,0.0,2000.000003,0,2.120920,1.8,10.0,0.000000,0.000000,0.0,0.0,199.999997,0,NaN,NaN,0,0
2,round0_GES-10,1.000000,0.0,2000.000003,0,1.682654,1.8,10.0,0.000000,80.137303,0.0,0.0,190.847281,0,NaN,NaN,0,0
3,round0_GES-13,8.613216,0.0,2000.000003,0,1.530000,1.8,10.0,0.000000,79.615557,0.0,0.0,199.999997,0,NaN,NaN,0,0


First, pick only the monochromator reads on Cytation 3, and then drop read information:

In [255]:
df.groupby(['Reader','Read']).Read.count()

Reader     Read      
Cytation5  GFP-M-Gext    36
Name: Read, dtype: int64

Drop columns that aren't going to be used in the fitting:

In [256]:
df.columns

Index(['experiment_id', 'Plate', 'Well', 'Read', 'Experiment', 'Name', 'Type',
       '[DNA] (nM)', '[PEG4K 40%] (%)', '[RNAse Inhib] (U/mL)',
       '[PMix] (mg/mL)', '[Ribosome] (uM)', 'Rxn Volume (uL)',
       '[Magnesium acetate] (mM)', '[Creatine phosphate] (mM)',
       '[Potassium glutamate] (mM)', '[HEPES] (mM)', '[PPK] (uM)',
       '[PolyP] (mM)', '[ATP] (mM)', '[GTP] (mM)', '[CTP] (mM)', '[UTP] (mM)',
       '[TCEP] (mM)', '[Folinic acid] (mM)', '[Spermidine] (mM)',
       '[Amino acid mix] (mM)', '[tRNA] (ug/uL)', 'IsNEB', 'Product',
       'HasMgAR953', 'Date', 'Reader', 'Gain', 'Read Type', 'Read_new',
       'sigmoid_steady_state (ng/uL)', 'sigmoid_rate (1/h)',
       'sigmoid_time_offset (h)', 'drift_rate (ng/uL/h)',
       'drift_rate_time_offset (h)', 'success'],
      dtype='object')

In [338]:
## Convert categorical "Product" to one hot (which is just plamGFP or not (= deGFP))
product_bool = (df["Product"] == "plamGFP").astype(bool)
# product_bool.loc[0] = "feature"
df["isplamGFP"] = product_bool
# df.loc[0, "isplamGFP"] = "feature"
df = df.drop("Product", axis=1)

In [339]:
df.drop(['Plate', 'Read', 'Well', 'Type','Experiment', 'Name',
         'Date', 'Reader', 'Gain', 'Read Type', 'Read_new', 'drift_rate (ng/uL/h)', 'drift_rate_time_offset (h)'
       ],axis=1, inplace=True) #  'Rxn Volume (uL)'

In [340]:
df_merged = pd.concat([df, prev_dataset.loc[1:,:], ges_known.loc[1:,:]], ignore_index=True)

In [341]:
df_merged

,experiment_id,[DNA] (nM),[PEG4K 40%] (%),[RNAse Inhib] (U/mL),[PMix] (mg/mL),[Ribosome] (uM),Rxn Volume (uL),[Magnesium acetate] (mM),[Creatine phosphate] (mM),[Potassium glutamate] (mM),...,[Spermidine] (mM),[Amino acid mix] (mM),[tRNA] (ug/uL),IsNEB,HasMgAR953,sigmoid_steady_state (ng/uL),sigmoid_rate (1/h),sigmoid_time_offset (h),success,isplamGFP
0,20260309-round1_G8,2.996187,0.0,2000.000003,1.833326,1.8,10.0,12.469,0.000000,100.000000,...,2.0,0.3,3.5,0.0,1.0,3.370488,4.565878,0.839765,1,0
1,20260309-round1_G9,1.000000,0.0,2000.000003,1.555191,1.8,10.0,24.969,71.000000,122.500000,...,2.0,0.3,3.5,0.0,1.0,12.284859,2.318723,1.254914,1,0
2,20260309-round1_G10,2.996187,0.0,2000.000003,1.950000,1.8,10.0,24.969,100.000000,60.000000,...,2.0,0.3,3.5,0.0,1.0,1.121951,3.976099,0.922998,1,0
3,20260309-round1_G11,2.996187,0.0,2000.000003,1.950000,1.8,10.0,12.469,49.000000,72.500000,...,2.0,0.3,3.5,0.0,1.0,1.598554,3.725096,0.922939,1,0
4,20260309-round1_G12,2.996187,0.0,2000.000003,1.950000,1.8,10.0,14.969,20.000000,140.000000,...,2.0,0.3,3.5,0.0,1.0,46.387345,3.044413,1.116157,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,Mg_K_screen-20251105_090558_P3,5.000000,0.0,2000.000000,1.800000,1.8,5.0,7.000,20.000000,140.000000,...,2.0,0.3,3.5,1.0,NaN,44.246019,2.091836,1.455216,1,1
178,Mg_K_screen-20251105_090558_P4,5.000000,0.0,2000.000000,1.800000,1.8,5.0,9.000,20.000000,100.000000,...,2.0,0.3,3.5,1.0,NaN,20.547366,2.272599,1.365400,1,1
179,round0_GES-9,19.999999,0.0,2000.000003,2.120920,1.8,10.0,0.000,0.000000,199.999997,...,NaN,NaN,NaN,0.0,NaN,0.000000,NaN,NaN,0,0
180,round0_GES-10,1.000000,0.0,2000.000003,1.682654,1.8,10.0,0.000,80.137303,190.847281,...,NaN,NaN,NaN,0.0,NaN,0.000000,NaN,NaN,0,0


In [342]:
df_merged.HasMgAR953 = df_merged.HasMgAR953.fillna(0)

In [343]:
df_merged[['success', 'IsNEB', 'HasMgAR953', 'isplamGFP']] = df_merged[['success', 'IsNEB', 'HasMgAR953', 'isplamGFP']].astype('boolean')

In [344]:
df_merged

,experiment_id,[DNA] (nM),[PEG4K 40%] (%),[RNAse Inhib] (U/mL),[PMix] (mg/mL),[Ribosome] (uM),Rxn Volume (uL),[Magnesium acetate] (mM),[Creatine phosphate] (mM),[Potassium glutamate] (mM),...,[Spermidine] (mM),[Amino acid mix] (mM),[tRNA] (ug/uL),IsNEB,HasMgAR953,sigmoid_steady_state (ng/uL),sigmoid_rate (1/h),sigmoid_time_offset (h),success,isplamGFP
0,20260309-round1_G8,2.996187,0.0,2000.000003,1.833326,1.8,10.0,12.469,0.000000,100.000000,...,2.0,0.3,3.5,False,True,3.370488,4.565878,0.839765,True,False
1,20260309-round1_G9,1.000000,0.0,2000.000003,1.555191,1.8,10.0,24.969,71.000000,122.500000,...,2.0,0.3,3.5,False,True,12.284859,2.318723,1.254914,True,False
2,20260309-round1_G10,2.996187,0.0,2000.000003,1.950000,1.8,10.0,24.969,100.000000,60.000000,...,2.0,0.3,3.5,False,True,1.121951,3.976099,0.922998,True,False
3,20260309-round1_G11,2.996187,0.0,2000.000003,1.950000,1.8,10.0,12.469,49.000000,72.500000,...,2.0,0.3,3.5,False,True,1.598554,3.725096,0.922939,True,False
4,20260309-round1_G12,2.996187,0.0,2000.000003,1.950000,1.8,10.0,14.969,20.000000,140.000000,...,2.0,0.3,3.5,False,True,46.387345,3.044413,1.116157,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,Mg_K_screen-20251105_090558_P3,5.000000,0.0,2000.000000,1.800000,1.8,5.0,7.000,20.000000,140.000000,...,2.0,0.3,3.5,True,False,44.246019,2.091836,1.455216,True,True
178,Mg_K_screen-20251105_090558_P4,5.000000,0.0,2000.000000,1.800000,1.8,5.0,9.000,20.000000,100.000000,...,2.0,0.3,3.5,True,False,20.547366,2.272599,1.365400,True,True
179,round0_GES-9,19.999999,0.0,2000.000003,2.120920,1.8,10.0,0.000,0.000000,199.999997,...,NaN,NaN,NaN,False,False,0.000000,NaN,NaN,False,False
180,round0_GES-10,1.000000,0.0,2000.000003,1.682654,1.8,10.0,0.000,80.137303,190.847281,...,NaN,NaN,NaN,False,False,0.000000,NaN,NaN,False,False


In [345]:
df_merged.HasMgAR953.value_counts()

HasMgAR953
False    146
True      36
Name: count, dtype: Int64

In [346]:
df_merged.isplamGFP.value_counts()

isplamGFP
True     137
False     45
Name: count, dtype: Int64

In [283]:
# Check if we have columns with different names
prev_dataset.columns[~prev_dataset.columns.isin(df.columns)]

Index([], dtype='object')

In [284]:
df.columns[~df.columns.isin(prev_dataset.columns)]

Index(['[HEPES] (mM)', '[ATP] (mM)', '[GTP] (mM)', '[CTP] (mM)', '[UTP] (mM)',
       '[TCEP] (mM)', '[Folinic acid] (mM)', '[Spermidine] (mM)',
       '[Amino acid mix] (mM)', '[tRNA] (ug/uL)', 'HasMgAR953'],
      dtype='object')

### Remove failures

In [347]:
failure_inds = df_merged.success == False

In [348]:
df_merged.loc[failure_inds,'sigmoid_steady_state (ng/uL)'] = 0
df_merged.loc[failure_inds,'sigmoid_rate (1/h)'] = np.nan
df_merged.loc[failure_inds,'sigmoid_time_offset (h)'] = np.nan

In [349]:
df_merged

,experiment_id,[DNA] (nM),[PEG4K 40%] (%),[RNAse Inhib] (U/mL),[PMix] (mg/mL),[Ribosome] (uM),Rxn Volume (uL),[Magnesium acetate] (mM),[Creatine phosphate] (mM),[Potassium glutamate] (mM),...,[Spermidine] (mM),[Amino acid mix] (mM),[tRNA] (ug/uL),IsNEB,HasMgAR953,sigmoid_steady_state (ng/uL),sigmoid_rate (1/h),sigmoid_time_offset (h),success,isplamGFP
0,20260309-round1_G8,2.996187,0.0,2000.000003,1.833326,1.8,10.0,12.469,0.000000,100.000000,...,2.0,0.3,3.5,False,True,3.370488,4.565878,0.839765,True,False
1,20260309-round1_G9,1.000000,0.0,2000.000003,1.555191,1.8,10.0,24.969,71.000000,122.500000,...,2.0,0.3,3.5,False,True,12.284859,2.318723,1.254914,True,False
2,20260309-round1_G10,2.996187,0.0,2000.000003,1.950000,1.8,10.0,24.969,100.000000,60.000000,...,2.0,0.3,3.5,False,True,1.121951,3.976099,0.922998,True,False
3,20260309-round1_G11,2.996187,0.0,2000.000003,1.950000,1.8,10.0,12.469,49.000000,72.500000,...,2.0,0.3,3.5,False,True,1.598554,3.725096,0.922939,True,False
4,20260309-round1_G12,2.996187,0.0,2000.000003,1.950000,1.8,10.0,14.969,20.000000,140.000000,...,2.0,0.3,3.5,False,True,46.387345,3.044413,1.116157,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,Mg_K_screen-20251105_090558_P3,5.000000,0.0,2000.000000,1.800000,1.8,5.0,7.000,20.000000,140.000000,...,2.0,0.3,3.5,True,False,44.246019,2.091836,1.455216,True,True
178,Mg_K_screen-20251105_090558_P4,5.000000,0.0,2000.000000,1.800000,1.8,5.0,9.000,20.000000,100.000000,...,2.0,0.3,3.5,True,False,20.547366,2.272599,1.365400,True,True
179,round0_GES-9,19.999999,0.0,2000.000003,2.120920,1.8,10.0,0.000,0.000000,199.999997,...,NaN,NaN,NaN,False,False,0.000000,NaN,NaN,False,False
180,round0_GES-10,1.000000,0.0,2000.000003,1.682654,1.8,10.0,0.000,80.137303,190.847281,...,NaN,NaN,NaN,False,False,0.000000,NaN,NaN,False,False


In [350]:
column_types = pd.DataFrame(data=np.reshape(["feature",]*len(df_merged.columns), (1, len(df_merged.columns))), columns=df_merged.columns)

In [351]:
column_types.experiment_id.iloc[0] = 'id'
column_types.success.iloc[0] = 'classifier'
column_types.loc[0,'sigmoid_steady_state (ng/uL)':'sigmoid_time_offset (h)'] = 'regressor'

In [352]:
column_types

,experiment_id,[DNA] (nM),[PEG4K 40%] (%),[RNAse Inhib] (U/mL),[PMix] (mg/mL),[Ribosome] (uM),Rxn Volume (uL),[Magnesium acetate] (mM),[Creatine phosphate] (mM),[Potassium glutamate] (mM),...,[Spermidine] (mM),[Amino acid mix] (mM),[tRNA] (ug/uL),IsNEB,HasMgAR953,sigmoid_steady_state (ng/uL),sigmoid_rate (1/h),sigmoid_time_offset (h),success,isplamGFP
0,id,feature,feature,feature,feature,feature,feature,feature,feature,feature,...,feature,feature,feature,feature,feature,regressor,regressor,regressor,classifier,feature


In [353]:
# Save columns with 0 variance
pd.concat([column_types, df_merged],axis=0,ignore_index=True).to_csv("20260309-round-1-allcols.csv", index=False)

In [354]:
# Drop columns that have only constant values
unique_cols = df_merged.columns[df_merged[1:].nunique() == 1]
df_unique = df_merged.drop(unique_cols, axis=1)

In [355]:
unique_cols

Index(['[HEPES] (mM)', '[ATP] (mM)', '[GTP] (mM)', '[CTP] (mM)', '[UTP] (mM)',
       '[TCEP] (mM)', '[Folinic acid] (mM)', '[Spermidine] (mM)',
       '[Amino acid mix] (mM)', '[tRNA] (ug/uL)'],
      dtype='object')

In [367]:
column_types.columns[~column_types.columns.isin(unique_cols)]

Index(['experiment_id', '[DNA] (nM)', '[PEG4K 40%] (%)',
       '[RNAse Inhib] (U/mL)', '[PMix] (mg/mL)', '[Ribosome] (uM)',
       'Rxn Volume (uL)', '[Magnesium acetate] (mM)',
       '[Creatine phosphate] (mM)', '[Potassium glutamate] (mM)', '[PPK] (uM)',
       '[PolyP] (mM)', 'IsNEB', 'HasMgAR953', 'sigmoid_steady_state (ng/uL)',
       'sigmoid_rate (1/h)', 'sigmoid_time_offset (h)', 'success',
       'isplamGFP'],
      dtype='object')

In [368]:
pd.concat([column_types[column_types.columns[~column_types.columns.isin(unique_cols)]], df_unique],axis=0,ignore_index=True)

,experiment_id,[DNA] (nM),[PEG4K 40%] (%),[RNAse Inhib] (U/mL),[PMix] (mg/mL),[Ribosome] (uM),Rxn Volume (uL),[Magnesium acetate] (mM),[Creatine phosphate] (mM),[Potassium glutamate] (mM),[PPK] (uM),[PolyP] (mM),IsNEB,HasMgAR953,sigmoid_steady_state (ng/uL),sigmoid_rate (1/h),sigmoid_time_offset (h),success,isplamGFP
0,id,feature,feature,feature,feature,feature,feature,feature,feature,feature,feature,feature,feature,feature,regressor,regressor,regressor,classifier,feature
1,20260309-round1_G8,2.996187,0.0,2000.000003,1.833326,1.8,10.0,12.469,0.0,100.0,0.0,0.0,False,True,3.370488,4.565878,0.839765,True,False
2,20260309-round1_G9,1.0,0.0,2000.000003,1.555191,1.8,10.0,24.969,71.0,122.5,0.0,0.0,False,True,12.284859,2.318723,1.254914,True,False
3,20260309-round1_G10,2.996187,0.0,2000.000003,1.95,1.8,10.0,24.969,100.0,60.0,0.0,0.0,False,True,1.121951,3.976099,0.922998,True,False
4,20260309-round1_G11,2.996187,0.0,2000.000003,1.95,1.8,10.0,12.469,49.0,72.5,0.0,0.0,False,True,1.598554,3.725096,0.922939,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178,Mg_K_screen-20251105_090558_P3,5.0,0.0,2000.0,1.8,1.8,5.0,7.0,20.0,140.0,0.0,0.0,True,False,44.246019,2.091836,1.455216,True,True
179,Mg_K_screen-20251105_090558_P4,5.0,0.0,2000.0,1.8,1.8,5.0,9.0,20.0,100.0,0.0,0.0,True,False,20.547366,2.272599,1.3654,True,True
180,round0_GES-9,19.999999,0.0,2000.000003,2.12092,1.8,10.0,0.0,0.0,199.999997,0.0,0.0,False,False,0.0,NaN,NaN,False,False
181,round0_GES-10,1.0,0.0,2000.000003,1.682654,1.8,10.0,0.0,80.137303,190.847281,0.0,0.0,False,False,0.0,NaN,NaN,False,False


In [369]:
pd.concat([column_types[column_types.columns[~column_types.columns.isin(unique_cols)]], df_unique],axis=0,ignore_index=True).to_csv("20260309-round-1-igor.csv", index=False)

For setting bounds, keep track of maximum and minimum values:

In [329]:
df_merged.query("experiment_id == 'PPK_20250616_G1' or experiment_id == 'PMix-MFG-128-PURE-Activity-QC_D2' ").loc[:, ['experiment_id', '[DNA] (nM)', '[PEG4K 40%] (%)',
       '[RNAse Inhib] (U/mL)', '[PMix] (mg/mL)', '[Ribosome] (uM)',
       'Rxn Volume (uL)', '[Magnesium acetate] (mM)',
       '[Creatine phosphate] (mM)', '[Potassium glutamate] (mM)',
       '[HEPES] (mM)', '[PPK] (uM)', '[PolyP] (mM)', 'IsNEB']]

,experiment_id,[DNA] (nM),[PEG4K 40%] (%),[RNAse Inhib] (U/mL),[PMix] (mg/mL),[Ribosome] (uM),Rxn Volume (uL),[Magnesium acetate] (mM),[Creatine phosphate] (mM),[Potassium glutamate] (mM),[HEPES] (mM),[PPK] (uM),[PolyP] (mM),IsNEB
58,PMix-MFG-128-PURE-Activity-QC_D2,3.206522,0.0,2000.0,1.8,1.8,10.0,8.0,20.0,100.0,50.0,0.0,0.0,False
116,PPK_20250616_G1,2.996187,2.0,0.0,1.8,1.8,10.0,18.0,20.0,100.0,50.0,2.0,30.0,True


In [232]:
pd.to_numeric(df_merged.query('`[PPK] (uM)` == 0').loc[:,'sigmoid_steady_state (ng/uL)']).describe()

count    152.000000
mean      28.095345
std       26.647120
min        0.000000
25%        2.095126
50%       26.278163
75%       44.337920
max      103.773184
Name: sigmoid_steady_state (ng/uL), dtype: float64

In [233]:
pd.to_numeric(df_merged.query('`[PPK] (uM)` > 0').loc[:,'sigmoid_steady_state (ng/uL)']).describe()

count     30.000000
mean      21.716153
std       32.762438
min        0.027403
25%        0.047231
50%       10.983892
75%       27.926717
max      114.117675
Name: sigmoid_steady_state (ng/uL), dtype: float64

In [330]:
numerical_vals_df = df_merged.loc[:, '[DNA] (nM)':].apply(pd.to_numeric)

In [331]:
minmax_df = pd.DataFrame({'min': numerical_vals_df.min(axis=0), 'max': numerical_vals_df.max(axis=0)}).T # 

In [332]:
minmax_df

,[DNA] (nM),[PEG4K 40%] (%),[RNAse Inhib] (U/mL),[PMix] (mg/mL),[Ribosome] (uM),Rxn Volume (uL),[Magnesium acetate] (mM),[Creatine phosphate] (mM),[Potassium glutamate] (mM),[HEPES] (mM),...,[Spermidine] (mM),[Amino acid mix] (mM),[tRNA] (ug/uL),IsNEB,HasMgAR953,sigmoid_steady_state (ng/uL),sigmoid_rate (1/h),sigmoid_time_offset (h),success,isplamGFP
min,1.0,0.0,0.0,1.53,1.8,5.0,0.0,0.0,60.0,50.0,...,2.0,0.3,3.5,False,False,0.0,1.367521,0.589683,False,False
max,19.999999,2.0,2000.000003,2.12092,3.24,10.0,24.969,100.0,200.0,50.0,...,2.0,0.3,3.5,True,True,114.117675,5.097274,2.589723,True,True


In [333]:
minmax_df.to_csv("20260309-round-1-min-max.csv")